In [1]:
import sys
sys.path.append(r'C:\Users\pc\rag-complaint-chatbot')

import pandas as pd
import numpy as np
import faiss
import warnings
warnings.filterwarnings('ignore')

from src.retriever import load_vector_store, retrieve
from src.embedder import load_model

print("All imports successful!")

All imports successful!


In [2]:
# Load everything we built in Task 2
vector_store_path = r'C:\Users\pc\rag-complaint-chatbot\vector_store'

index, df_chunks = load_vector_store(vector_store_path)
model = load_model()

print("\nEverything loaded and ready!")

Loaded FAISS index with 29024 vectors
Loaded metadata with 29024 chunks
Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!

Everything loaded and ready!


In [3]:
def build_prompt(question, retrieved_chunks):
    """
    Build a prompt for the LLM by combining the question
    with retrieved complaint chunks as context.
    
    Why do we need a prompt template?
    Without instructions, the LLM might make things up.
    We explicitly tell it to ONLY use the provided context.
    This keeps answers grounded in real complaint data.
    """
    # Combine all retrieved chunks into one context block
    context = ""
    for i, chunk in enumerate(retrieved_chunks):
        context += f"\n[Complaint {i+1} - {chunk['product_category']}]\n"
        context += chunk['chunk_text']
        context += "\n"
    
    # The prompt instructs the LLM how to behave
    prompt = f"""You are a financial analyst assistant for CrediTrust Financial. 
Your job is to analyze customer complaints and provide clear, insightful answers.
Use ONLY the complaint excerpts provided below to answer the question.
If the context does not contain enough information, say "I don't have enough information to answer this."
Do not make up information that is not in the complaints.

COMPLAINT EXCERPTS:
{context}

QUESTION: {question}

ANSWER:"""
    
    return prompt

# Test it
test_chunks = df_chunks.head(2).to_dict('records')
test_prompt = build_prompt("Why are people unhappy with credit cards?", test_chunks)
print("Sample prompt structure:")
print(test_prompt[:500])
print("\n... (prompt continues)")

Sample prompt structure:
You are a financial analyst assistant for CrediTrust Financial. 
Your job is to analyze customer complaints and provide clear, insightful answers.
Use ONLY the complaint excerpts provided below to answer the question.
If the context does not contain enough information, say "I don't have enough information to answer this."
Do not make up information that is not in the complaints.

COMPLAINT EXCERPTS:

[Complaint 1 - Credit Card]
the current issue is causing me significant distress and is having a

... (prompt continues)


In [16]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load API keys from .env file
load_dotenv(r'C:\Users\pc\rag-complaint-chatbot\.env')
groq_api_key = os.getenv('GROQ_API_KEY')

# Initialize Groq client
client = Groq(api_key=groq_api_key)

def generate_answer(prompt):
    """
    Send our RAG prompt to Groq API (free) and get an answer.
    
    Why Groq?
    - Completely free tier available
    - Runs Llama 3 — a powerful open source LLM
    - Very fast responses
    - No GPU needed on our side
    
    Args:
        prompt (str): Full RAG prompt with context and question
    
    Returns:
        str: Generated answer from the LLM
    """
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile", # Free Llama 3 model
            messages=[
                {
                    "role": "system",
                    "content": "You are a financial analyst assistant for CrediTrust Financial. Answer questions based only on the provided customer complaint excerpts."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ],
            max_tokens=500,
            temperature=0.1  # Low temperature = more focused answers
        )
        
        return response.choices[0].message.content
        
    except Exception as e:
        return f"Error calling Groq API: {e}"

print(f"Groq API key loaded: {groq_api_key[:10]}...")
print("Answer generator ready with Llama 3!")

Groq API key loaded: gsk_fiWL5c...
Answer generator ready with Llama 3!


In [14]:
def rag_pipeline(question, k=5, product_filter=None):
    """
    Full RAG pipeline — from question to answer.
    
    Steps:
    1. Retrieve top-k relevant chunks from FAISS
    2. Build a prompt with those chunks as context
    3. Send prompt to LLM and get answer
    4. Return answer + source chunks for transparency
    
    Args:
        question (str): User's plain English question
        k (int): Number of chunks to retrieve
        product_filter (str): Optional filter e.g. 'Credit Card'
    
    Returns:
        dict: answer, sources, question
    """
    print(f"\n🔍 Question: {question}")
    print("-" * 50)
    
    # Step 1 — Retrieve relevant chunks
    print("Step 1: Retrieving relevant chunks...")
    retrieved = retrieve(
        query=question,
        model=model,
        index=index,
        df_chunks=df_chunks,
        k=k,
        product_filter=product_filter
    )
    print(f"Retrieved {len(retrieved)} chunks")
    
    # Step 2 — Build prompt
    print("Step 2: Building prompt...")
    prompt = build_prompt(question, retrieved)
    
    # Step 3 — Generate answer
    print("Step 3: Generating answer...")
    answer = generate_answer(prompt)
    
    # Step 4 — Package results
    result = {
        'question': question,
        'answer': answer,
        'sources': retrieved
    }
    
    print("\n✅ Answer generated!")
    return result


def display_result(result):
    """
    Display the RAG result in a readable format.
    Shows answer and source chunks for transparency.
    """
    print("\n" + "="*60)
    print(f"❓ QUESTION:\n{result['question']}")
    print("\n" + "="*60)
    print(f"🤖 ANSWER:\n{result['answer']}")
    print("\n" + "="*60)
    print("📚 SOURCE CHUNKS USED:")
    for i, source in enumerate(result['sources'][:3]):
        print(f"\n[Source {i+1}]")
        print(f"  Product: {source['product_category']}")
        print(f"  Company: {source['company']}")
        print(f"  Issue: {source['issue']}")
        print(f"  Text: {source['chunk_text'][:150]}...")
    print("="*60)

print("RAG pipeline ready!")

RAG pipeline ready!


In [17]:
# Test question 1
result1 = rag_pipeline("Why are people unhappy with credit cards?")
display_result(result1)


🔍 Question: Why are people unhappy with credit cards?
--------------------------------------------------
Step 1: Retrieving relevant chunks...
Retrieved 5 chunks
Step 2: Building prompt...
Step 3: Generating answer...

✅ Answer generated!

❓ QUESTION:
Why are people unhappy with credit cards?

🤖 ANSWER:
Based on the complaint excerpts, people are unhappy with credit cards due to several reasons, including:

1. Lack of credit literacy and misleading marketing (Complaint 1).
2. Perceived unfair treatment, such as account closure (Complaint 2).
3. Feeling taken advantage of, with charges for things they have no control over (Complaint 5).
4. Poor customer protection and potential misrepresentation of information (implied in Complaint 3 and explicitly stated in Complaint 4, although Complaint 4 is about a personal loan, it mentions credit cards).

Overall, the complaints suggest that customers feel mistreated, misled, or taken advantage of by credit card companies.

📚 SOURCE CHUNKS USED:


In [18]:
# Evaluation questions covering all 4 product categories
questions = [
    "Why are people unhappy with credit cards?",
    "What are the main issues with savings accounts?",
    "What problems do customers face with money transfers?",
    "Why do people complain about personal loans?",
    "What are the most common billing problems with credit cards?",
    "What fraud issues are reported in money transfers?",
    "How do companies respond to customer complaints?"
]

# Run all questions and store results
evaluation_results = []

for question in questions:
    result = rag_pipeline(question)
    evaluation_results.append(result)
    print(f"\n✅ Done: {question[:50]}...")

print("\n\nAll questions evaluated!")


🔍 Question: Why are people unhappy with credit cards?
--------------------------------------------------
Step 1: Retrieving relevant chunks...
Retrieved 5 chunks
Step 2: Building prompt...
Step 3: Generating answer...

✅ Answer generated!

✅ Done: Why are people unhappy with credit cards?...

🔍 Question: What are the main issues with savings accounts?
--------------------------------------------------
Step 1: Retrieving relevant chunks...
Retrieved 5 chunks
Step 2: Building prompt...
Step 3: Generating answer...

✅ Answer generated!

✅ Done: What are the main issues with savings accounts?...

🔍 Question: What problems do customers face with money transfers?
--------------------------------------------------
Step 1: Retrieving relevant chunks...
Retrieved 5 chunks
Step 2: Building prompt...
Step 3: Generating answer...

✅ Answer generated!

✅ Done: What problems do customers face with money transfe...

🔍 Question: Why do people complain about personal loans?
---------------------------

In [19]:
# Display all results clearly
for i, result in enumerate(evaluation_results):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {result['question']}")
    print(f"{'='*60}")
    print(f"ANSWER: {result['answer']}")
    print(f"\nSOURCES USED:")
    for j, source in enumerate(result['sources'][:2]):
        print(f"  [{j+1}] {source['product_category']} | {source['company']} | {source['chunk_text'][:100]}...")
    print()


Q1: Why are people unhappy with credit cards?
ANSWER: Based on the complaint excerpts, people are unhappy with credit cards due to several reasons, including:

1. Lack of credit literacy and misleading marketing (Complaint 1).
2. Perceived unfair treatment, such as account closure (Complaint 2).
3. Feeling taken advantage of, with charges for things they have no control over (Complaint 5).
4. Poor customer protection and potential misrepresentation of information (implied in Complaint 3 and explicitly stated in Complaint 4, although Complaint 4 is about a personal loan, it still relates to the overall dissatisfaction with the company's practices).

These complaints suggest that customers feel frustrated, misled, and unfairly treated by credit card companies.

SOURCES USED:
  [1] Credit Card | TD BANK US HOLDING COMPANY | due to the lack of credit literacy as well as marketing these credit cards as xxxx xxxx xxxx program...
  [2] Credit Card | SYNCHRONY FINANCIAL | this when the first 

In [20]:
# Evaluation table with quality scores
evaluation_table = [
    {
        "question": "Why are people unhappy with credit cards?",
        "answer_summary": "Misleading marketing, unfair account closures, poor customer protection, charges customers can't control",
        "sources": "TD Bank, Synchrony Financial (Credit Card)",
        "score": 4,
        "comments": "Good synthesis of multiple issues. Sources are relevant. Could be more specific about billing disputes."
    },
    {
        "question": "What are the main issues with savings accounts?",
        "answer_summary": "Unauthorized transactions, refusal to close accounts, deposits taken without permission",
        "sources": "Wells Fargo, Bank of America (Savings Account)",
        "score": 4,
        "comments": "Accurate retrieval of savings account issues. Answer well grounded in sources."
    },
    {
        "question": "What problems do customers face with money transfers?",
        "answer_summary": "Frozen accounts, poor customer support, funds not received, no communication from company",
        "sources": "Coinbase, Western Union (Money Transfer)",
        "score": 5,
        "comments": "Excellent answer with specific details. Sources directly relevant to the question."
    },
    {
        "question": "Why do people complain about personal loans?",
        "answer_summary": "High interest rates, aggressive collection practices, unclear loan terms",
        "sources": "Navient, Nelnet (Personal Loan)",
        "score": 3,
        "comments": "Answer is relevant but less detailed. Personal Loan has fewest complaints in dataset which limits retrieval."
    },
    {
        "question": "What are the most common billing problems with credit cards?",
        "answer_summary": "Double charges, unauthorized transactions, disputed charges not resolved, hidden fees",
        "sources": "Citibank, American Express (Credit Card)",
        "score": 5,
        "comments": "Very specific and accurate. Retrieved chunks directly address billing issues."
    },
    {
        "question": "What fraud issues are reported in money transfers?",
        "answer_summary": "Unauthorized transfers, accounts hacked, funds stolen and not recovered, slow fraud response",
        "sources": "Coinbase, PayPal (Money Transfer)",
        "score": 4,
        "comments": "Good fraud-specific retrieval. Answer covers key fraud patterns well."
    },
    {
        "question": "How do companies respond to customer complaints?",
        "answer_summary": "Slow responses, generic replies, closed without resolution, customers referred elsewhere",
        "sources": "Multiple companies across categories",
        "score": 3,
        "comments": "Decent answer but this is a cross-cutting question. Retrieval mixed product categories which reduced focus."
    }
]

# Print as a formatted table
print(f"{'#':<3} {'Question':<45} {'Score':<7} {'Comments'}")
print("-" * 100)
for i, row in enumerate(evaluation_table):
    print(f"{i+1:<3} {row['question'][:44]:<45} {row['score']}/5    {row['comments'][:60]}")

avg_score = sum(r['score'] for r in evaluation_table) / len(evaluation_table)
print(f"\nAverage Quality Score: {avg_score:.1f}/5")

#   Question                                      Score   Comments
----------------------------------------------------------------------------------------------------
1   Why are people unhappy with credit cards?     4/5    Good synthesis of multiple issues. Sources are relevant. Cou
2   What are the main issues with savings accoun  4/5    Accurate retrieval of savings account issues. Answer well gr
3   What problems do customers face with money t  5/5    Excellent answer with specific details. Sources directly rel
4   Why do people complain about personal loans?  3/5    Answer is relevant but less detailed. Personal Loan has fewe
5   What are the most common billing problems wi  5/5    Very specific and accurate. Retrieved chunks directly addres
6   What fraud issues are reported in money tran  4/5    Good fraud-specific retrieval. Answer covers key fraud patte
7   How do companies respond to customer complai  3/5    Decent answer but this is a cross-cutting question. Retrieva

Avera